# Revisão do M4

Este notebook verifica o M4 atual e orienta sua refatoração conforme a RQ2 do paper_v8.

**Conclusão principal:** o cálculo legado é reproduzível, mas mistura contagem de commits já coberta pelo novo M3 com churn bruto severamente contaminado por dependências e artefatos gerados. O M4 revisado deve medir magnitude, intensidade e composição das mudanças limpas, sem duplicar a temporalidade de M3.

## Estrutura recomendada

| Saída | Pergunta respondida | Relação com M3 |
|---|---|---|
| **M4a — Magnitude de mudança limpa** | Quanto código-fonte/teste mudou? | Complementa “quando” com “quanto” |
| **M4b — Intensidade e amplitude da mudança** | As mudanças foram grandes por commit e espalhadas por quantos arquivos? | Não usa contagem de commits como resultado |
| **M4c — Composição do churn** | Quanto do churn vem de source/test versus dependências/gerados? | Audita a validade do volume observado |
| **M4d — Trajetória móvel de mudança limpa** | Quando a magnitude limpa e a amplitude de arquivos se acumulam? | Usa a temporalidade de M3, mas mede churn/arquivos, não autoria ou contagem de commits |

M3 permanece responsável por participação temporal dos commits e concentração de autoria. M4 não deve repetir `commit_n` como resultado principal.

## 1. Vínculo com a RQ2

> **RQ2:** How does the temporal density of repository activity contrast with the qualitative typification of human coordination friction across the project lifecycle?

O notebook extrai esse vínculo do paper atual e interrompe a execução se M4 deixar de pertencer à RQ2.

In [10]:
from hashlib import sha256
from pathlib import Path
import json
import re
import subprocess
import sys

import numpy as np
import pandas as pd
from IPython.display import HTML, display

PROJECT_ROOT = Path.cwd().parent
PAPER_PATH = PROJECT_ROOT / "paper_v8/latex_code/main.tex"
COMMITS_PATH = PROJECT_ROOT / "data/lake/git_commits.parquet"
FILES_PATH = PROJECT_ROOT / "data/lake/git_files.parquet"
SOURCE_PATH = PROJECT_ROOT / "data/analysis/code_churn_metrics.parquet"
M4_PATH = PROJECT_ROOT / "paper_v8/data/m4_repo_activity_density.csv"
PARENT_CACHE_DIR = PROJECT_ROOT / "data/raw/repos_parent_cache"

paper_text = PAPER_PATH.read_text(encoding="utf-8")
rq_matches = dict(
    re.findall(r"\\item \\textbf\{(RQ\d)[^}]*:\}\s*(.*?)\n", paper_text)
)
m4_position = paper_text.index(r"\textbf{M4 --")
rq2_position = paper_text.index(r"\subsubsection{RQ2:")
rq3_position = paper_text.index(r"\subsubsection{RQ3:")
detected_rq = "RQ2" if rq2_position < m4_position < rq3_position else "UNKNOWN"
assert detected_rq == "RQ2"
assert rq_matches.get(detected_rq)

paper_mapping_df = pd.DataFrame(
    [
        {
            "metric": "M4",
            "detected_rq": detected_rq,
            "rq_text": rq_matches[detected_rq],
            "paper_sha256": sha256(paper_text.encode("utf-8")).hexdigest(),
        }
    ]
)
paper_mapping_df

,metric,detected_rq,rq_text,paper_sha256
0,M4,RQ2,How does the temporal density of repository ac...,68bb2bbe3bd74611d7a2517844e943ca615845af21b8ec...


### 1.1 Configuração reproduzível

As decisões seguem os notebooks anteriores: mesma âncora por último voto T3, mirrors pais read-only, `committer date`, exclusão explícita de dependências/gerados e preservação do legado.

In [ ]:
CONFIG = {
    "metric": "M4",
    "expected_rq": "RQ2",
    "team_key": ["Semestre", "ID_Equipe"],
    "expected_team_semesters": 14,
    "legacy_cuts": ["T1", "T2", "T3"],
    "primary_anchor": "last_t3_evaluator_vote",
    "window_days_before": 7,
    "window_days_after": 7,
    "rolling_window_days": 7,
    "rolling_step_days": 1,
    "rolling_end_day_range_relative_to_t3": [-63, 7],
    "git_source": "read_only_parent_default_branch",
    "timestamp_field": "committer_date",
    "primary_outputs": [
        "clean_source_churn",
        "median_source_churn_per_touching_commit",
        "unique_clean_source_files",
        "clean_source_share_of_all_churn",
        "rolling_7d_clean_churn_trajectory",
    ],
    "excluded_path_tokens": [
        "node_modules/", ".venv/", "venv/", "env/", ".env/",
        "site-packages/", "vendor/", "dist/", "build/", ".next/",
        "coverage/", "__pycache__/", ".min.js",
    ],
    "commit_count_is_primary": False,
    "llm_calls_required": False,
    "export_outputs": False,
    "regression_tolerance": 1e-12,
}

REQUIRED_CONFIG_FIELDS = {
    "metric", "expected_rq", "team_key", "expected_team_semesters",
    "primary_anchor", "window_days_before", "window_days_after",
    "rolling_window_days", "rolling_step_days", "rolling_end_day_range_relative_to_t3",
    "git_source", "timestamp_field", "primary_outputs",
    "excluded_path_tokens", "commit_count_is_primary",
    "llm_calls_required", "regression_tolerance",
}
missing_config = REQUIRED_CONFIG_FIELDS - set(CONFIG)
assert not missing_config, sorted(missing_config)
assert CONFIG["expected_rq"] == detected_rq
assert CONFIG["commit_count_is_primary"] is False
CONFIG

{'metric': 'M4',
 'expected_rq': 'RQ2',
 'team_key': ['Semestre', 'ID_Equipe'],
 'expected_team_semesters': 14,
 'legacy_cuts': ['T1', 'T2', 'T3'],
 'primary_anchor': 'last_t3_evaluator_vote',
 'window_days_before': 7,
 'window_days_after': 7,
 'git_source': 'read_only_parent_default_branch',
 'timestamp_field': 'committer_date',
 'primary_outputs': ['clean_source_churn',
  'median_source_churn_per_touching_commit',
  'unique_clean_source_files',
  'clean_source_share_of_all_churn'],
 'excluded_path_tokens': ['node_modules/',
  '.venv/',
  'venv/',
  'env/',
  '.env/',
  'site-packages/',
  'vendor/',
  'dist/',
  'build/',
  '.next/',
  'coverage/',
  '__pycache__/',
  '.min.js'],
 'commit_count_is_primary': False,
 'llm_calls_required': False,
 'export_outputs': False,
 'regression_tolerance': 1e-12}

## 2. Auditoria do M4 legado

O fluxo legado é:

1. commits e arquivos no data lake;
2. `code_churn_metrics.parquet`, uma linha por equipe-semestre;
3. `m4_repo_activity_density.csv`, resumo por semestre, corte e métrica.

O M4 atual publica `cc_total` e `cc_commit_n`. A auditoria separa correção aritmética de validade da medida.

## 3. Contrato de relevância dos artefatos

O volume de mudança só é interpretável se os artefatos incluídos forem declarados. A política usa duas barreiras versionadas:

1. **Blacklist de caminhos:** dependências, ambientes, build e outros artefatos gerados têm precedência sobre a extensão. Um `.py` em `venv/` é `generated`, não `source`.
2. **Whitelist de extensões:** mesmo em `source` ou `test`, um arquivo só entra no M4 se sua extensão estiver na `SOURCE_CODE_EXTENSION_ALLOWLIST`. Portanto, `tests/README.md` e notebooks não entram por acidente.

| Categoria | Papel no diagnóstico | Entra em M4a/M4b? |
|---|---|---|
| `source` | Implementação do produto | Sim, se a extensão estiver na whitelist |
| `test` | Verificação automatizada | Sim, se a extensão estiver na whitelist |
| `planning`, `config`, `localization`, `asset`, `unknown` | Contexto ou escopo não equivalente a implementação | Não; reportar separadamente |
| `generated` | Dependências, ambientes, bundles, build e cobertura | Não; reportar como risco de contaminação |

A composição por categoria é uma saída obrigatória de auditoria, não uma medida de desempenho da equipe.

In [ ]:
import importlib.util

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pipeline_config import (
    FILE_CATEGORY_DEFINITION_VERSION,
    SOURCE_CODE_EXTENSION_ALLOWLIST,
    is_measurement_code_path,
)

cross_evidence_spec = importlib.util.spec_from_file_location(
    "cross_evidence_engine", PROJECT_ROOT / "08_cross_evidence_engine.py"
)
assert cross_evidence_spec and cross_evidence_spec.loader
cross_evidence_engine = importlib.util.module_from_spec(cross_evidence_spec)
cross_evidence_spec.loader.exec_module(cross_evidence_engine)
classify_file_category = cross_evidence_engine.classify_file_category

INCLUDED_M4_CATEGORIES = frozenset({"source", "test"})
M4_EXTENSION_ALLOWLIST = frozenset(SOURCE_CODE_EXTENSION_ALLOWLIST)
ARTIFACT_POLICY_VERSION = f"{FILE_CATEGORY_DEFINITION_VERSION}+m4-extension-allowlist-v1"


def is_m4_measurement_path(file_path: str, file_category: str) -> bool:
    return file_category in INCLUDED_M4_CATEGORIES and is_measurement_code_path(file_path)


SENTINEL_PATHS = pd.DataFrame(
    [
        ("src/app.py", ".py", "source", True),
        ("tests/test_app.py", ".py", "test", True),
        ("tests/README.md", ".md", "test", False),
        ("tests/exploration.ipynb", ".ipynb", "test", False),
        (".history/src/app_20251113000000.py", ".py", "generated", False),
        ("backend/testes/backups/llm_service.py", ".py", "generated", False),
        ("venv/lib/python3.11/site-packages/pkg/module.py", ".py", "generated", False),
        (".venv/lib/python3.11/site-packages/pkg/module.py", ".py", "generated", False),
        ("node_modules/react/index.js", ".js", "generated", False),
        ("dist/app.js", ".js", "generated", False),
        ("docs/architecture/schema.json", ".json", "planning", False),
        ("assets/logo.svg", ".svg", "asset", False),
    ],
    columns=["file_path", "file_extension", "expected_category", "expected_m4_inclusion"],
)

sentinel_classification = pd.concat(
    [
        SENTINEL_PATHS,
        pd.DataFrame(
            [
                classify_file_category(row.file_path, row.file_extension)
                for row in SENTINEL_PATHS.itertuples(index=False)
            ]
        ),
    ],
    axis=1,
)
sentinel_classification["extension_allowlisted"] = sentinel_classification["file_extension"].isin(M4_EXTENSION_ALLOWLIST)
sentinel_classification["included_in_m4"] = [
    is_m4_measurement_path(row.file_path, row.file_category)
    for row in sentinel_classification.itertuples(index=False)
]
assert sentinel_classification["file_category"].eq(sentinel_classification["expected_category"]).all()
assert sentinel_classification["included_in_m4"].eq(sentinel_classification["expected_m4_inclusion"]).all()
assert sentinel_classification["included_in_m4"].eq(sentinel_classification["extension_allowlisted"] & sentinel_classification["file_category"].isin(INCLUDED_M4_CATEGORIES)).all()
sentinel_classification

,file_path,file_extension,expected_category,expected_m4_inclusion,file_category,category_rule,category_confidence,category_warning,extension_allowlisted,included_in_m4
0,src/app.py,.py,source,True,source,extension:source:.py,1.0,None,True,True
1,tests/test_app.py,.py,test,True,test,path:test:tests/,1.0,None,True,True
2,tests/README.md,.md,test,False,test,path:test:tests/,1.0,None,False,False
3,tests/exploration.ipynb,.ipynb,test,False,test,path:test:tests/,1.0,None,False,False
4,venv/lib/python3.11/site-packages/pkg/module.py,.py,generated,False,generated,path:generated:site-packages/,0.8,generated_path_takes_precedence,True,False
5,.venv/lib/python3.11/site-packages/pkg/module.py,.py,generated,False,generated,path:generated:site-packages/,0.8,generated_path_takes_precedence,True,False
6,node_modules/react/index.js,.js,generated,False,generated,path:generated:node_modules/,0.8,generated_path_takes_precedence,True,False
7,dist/app.js,.js,generated,False,generated,path:generated:dist/,0.8,generated_path_takes_precedence,True,False
8,docs/architecture/schema.json,.json,planning,False,planning,path:planning,0.8,planning_path_takes_precedence_over_config_ext...,False,False
9,assets/logo.svg,.svg,asset,False,asset,extension:asset:.svg,1.0,None,False,False


### 3.1 Composição observada do churn

Esta auditoria usa todos os eventos `git_files` do lake para revelar o que o churn bruto representa. O M4 revisado só agrega eventos `source` e `test`; as demais categorias permanecem visíveis para que uma contaminação não seja silenciosamente removida.

In [ ]:
git_files = pd.read_parquet(FILES_PATH).copy()
REQUIRED_FILE_EVENT_COLUMNS = {
    "Semestre", "ID_Equipe", "temporal_marker", "file_path", "file_extension",
    "lines_added", "lines_deleted", "is_binary",
}
missing_file_event_columns = REQUIRED_FILE_EVENT_COLUMNS - set(git_files.columns)
assert not missing_file_event_columns, sorted(missing_file_event_columns)
assert not git_files.empty

classifications = pd.DataFrame(
    [
        classify_file_category(
            file_path=row.file_path,
            file_extension=row.file_extension,
            change_status=getattr(row, "change_status", None),
            file_path_old=getattr(row, "file_path_old", None),
        )
        for row in git_files.itertuples(index=False)
    ]
)
artifact_events = pd.concat([git_files.reset_index(drop=True), classifications], axis=1)
artifact_events["file_extension_normalized"] = artifact_events["file_extension"].fillna("").astype(str).str.lower()
artifact_events["extension_allowlisted"] = artifact_events["file_extension_normalized"].isin(M4_EXTENSION_ALLOWLIST)
artifact_events["included_in_m4"] = [
    is_m4_measurement_path(path, category)
    for path, category in zip(artifact_events["file_path"], artifact_events["file_category"])
]
artifact_events["lines_added_numeric"] = pd.to_numeric(artifact_events["lines_added"], errors="coerce").fillna(0)
artifact_events["lines_deleted_numeric"] = pd.to_numeric(artifact_events["lines_deleted"], errors="coerce").fillna(0)
artifact_events["churn_lines"] = artifact_events["lines_added_numeric"] + artifact_events["lines_deleted_numeric"]

artifact_composition = (
    artifact_events.groupby(
        ["Semestre", "temporal_marker", "file_category", "included_in_m4"],
        dropna=False,
    )
    .agg(
        file_event_n=("file_path", "size"),
        unique_path_n=("file_path", "nunique"),
        binary_event_n=("is_binary", "sum"),
        churn_lines=("churn_lines", "sum"),
        unknown_rule_n=("category_confidence", lambda values: int((values < 1).sum())),
    )
    .reset_index()
)
artifact_composition["churn_share_within_cut"] = artifact_composition["churn_lines"] / artifact_composition.groupby(
    ["Semestre", "temporal_marker"], dropna=False
)["churn_lines"].transform("sum").replace(0, np.nan)

artifact_policy_audit = (
    artifact_composition.groupby(["file_category", "included_in_m4"], dropna=False)
    .agg(
        file_event_n=("file_event_n", "sum"),
        unique_path_n=("unique_path_n", "sum"),
        churn_lines=("churn_lines", "sum"),
        low_confidence_n=("unknown_rule_n", "sum"),
    )
    .reset_index()
    .sort_values(["included_in_m4", "churn_lines"], ascending=[False, False])
)
assert artifact_events.loc[artifact_events["included_in_m4"], "file_category"].isin(INCLUDED_M4_CATEGORIES).all()
assert artifact_events.loc[artifact_events["included_in_m4"], "file_extension_normalized"].isin(M4_EXTENSION_ALLOWLIST).all()
assert artifact_events["file_category"].notna().all()
artifact_policy_audit

,file_category,included_in_m4,file_event_n,unique_path_n,churn_lines,low_confidence_n
4,source,True,4410,3074,321132.0,0
5,test,True,13,12,1314.0,0
2,generated,False,43894,22520,13115489.0,22648
1,config,False,486,255,216607.0,0
3,planning,False,1466,1312,93098.0,65
6,unknown,False,336,179,46635.0,336
0,asset,False,436,350,869.0,0


### 3.2 Portão de qualidade e fila de revisão

A exclusão é deliberadamente conservadora: `unknown` não entra no M4. Porém, não deve ser ignorado. O portão abaixo falha se uma categoria não permitida entrar no cálculo e a fila lista os caminhos desconhecidos com maior churn para revisão e eventual atualização versionada da taxonomia.

In [9]:
ineligible_events = artifact_events.loc[~artifact_events["included_in_m4"]]
assert artifact_events.loc[
    artifact_events["included_in_m4"], "file_category"
].isin(INCLUDED_M4_CATEGORIES).all()
assert artifact_events.loc[
    artifact_events["included_in_m4"], "file_extension_normalized"
].isin(M4_EXTENSION_ALLOWLIST).all()
assert not artifact_events.loc[
    artifact_events["file_path"].astype(str).str.contains(
        r"(?:^|/)(?:\.venv|venv|\.env|env|node_modules|site-packages)(?:/|$)",
        regex=True,
    ),
    "included_in_m4",
].any()

unknown_review_queue = (
    artifact_events.loc[artifact_events["file_category"].eq("unknown")]
    .groupby(["file_path", "file_extension", "category_rule"], dropna=False)
    .agg(
        file_event_n=("file_path", "size"),
        churn_lines=("churn_lines", "sum"),
        semesters=("Semestre", lambda values: ", ".join(sorted(map(str, set(values))))),
    )
    .reset_index()
    .sort_values(["churn_lines", "file_event_n"], ascending=False)
)

artifact_quality_gate = pd.DataFrame(
    [
        ("artifact_policy_version", ARTIFACT_POLICY_VERSION),
        ("included_categories", ", ".join(sorted(INCLUDED_M4_CATEGORIES))),
        ("allowed_extensions_n", len(M4_EXTENSION_ALLOWLIST)),
        ("generated_events_included", int(((artifact_events["file_category"] == "generated") & artifact_events["included_in_m4"]).sum())),
        ("non_allowlisted_events_included", int((artifact_events["included_in_m4"] & ~artifact_events["extension_allowlisted"]).sum())),
        ("unknown_events_included", int(((artifact_events["file_category"] == "unknown") & artifact_events["included_in_m4"]).sum())),
        ("unknown_churn_lines_excluded", float(unknown_review_queue["churn_lines"].sum())),
        ("unknown_paths_requiring_review", int(len(unknown_review_queue))),
    ],
    columns=["check", "value"],
)
display(artifact_quality_gate)
unknown_review_queue.head(20)

,check,value
0,artifact_policy_version,file-category-rules-v2+m4-extension-allowlist-v1
1,included_categories,"source, test"
2,allowed_extensions_n,25
3,generated_events_included,0
4,non_allowlisted_events_included,0
5,unknown_events_included,0
6,unknown_churn_lines_excluded,46635.0
7,unknown_paths_requiring_review,127


,file_path,file_extension,category_rule,file_event_n,churn_lines,semesters
59,apps/backend/benchmarking/analysis.ipynb,.ipynb,extension:unknown:.ipynb,5,16802.0,2025.2
56,apps/backend/backend/benchmarking/analysis.ipynb,.ipynb,extension:unknown:.ipynb,2,11898.0,2025.2
74,arquivos/Arq_Bracis.log,.log,extension:unknown:.log,2,3026.0,2025.2
94,backend/funcionalidades/arquivos/1706.03762v7.tex,.tex,extension:unknown:.tex,3,1473.0,2025.2
97,backend/funcionalidades/arquivos/1706.03762v7_...,.tex,extension:unknown:.tex,3,1452.0,2025.2
96,backend/funcionalidades/arquivos/1706.03762v7_...,.tex,extension:unknown:.tex,2,1048.0,2025.2
5,.gitignore,,extension:empty,50,885.0,"2025.2, 2026.1"
75,arquivos/Arq_Bracis.tex,.tex,extension:unknown:.tex,2,852.0,2025.2
9,.history/.gitignore_20251110101342,,extension:empty,2,596.0,2025.2
10,.history/.gitignore_20251110101356,,extension:empty,2,596.0,2025.2


## 4. Trajetória móvel de mudança limpa

M4d aplica a mesma série diária de janelas retrospectivas de 7 dias usada em M3/M7, alinhada ao último voto T3 de cada equipe. Cada janela soma somente churn de arquivos `source`/`test` que passaram pelo contrato de categorias e whitelist; também conta caminhos únicos alterados. Assim, a curva mede magnitude e amplitude da mudança limpa, não frequência de commits.

In [ ]:
from pipeline_config import EVALUATOR_TEMPORAL_CUTS
from pipeline_core import _git_diff_paths


def last_t3_evaluator_vote_anchors(project_root: Path) -> pd.DataFrame:
    rows = []
    for semester, checkpoint_ranges in EVALUATOR_TEMPORAL_CUTS.items():
        votes = pd.read_csv(project_root / f"data/processed/forms/{semester}/avaliadores.csv")
        votes["ID_Equipe"] = (
            votes["To which group do these scores refer?"]
            .str.extract(r"Group\s+(\d+)", expand=False)
            .astype(int)
            .map(lambda value: f"TEAM_{value:02d}")
        )
        votes["vote_at"] = pd.to_datetime(votes["Timestamp"], format="mixed").dt.tz_localize("America/Fortaleza")
        start_date, end_date = checkpoint_ranges["T3"]
        in_t3 = votes.loc[votes["vote_at"].dt.date.between(pd.Timestamp(start_date).date(), pd.Timestamp(end_date).date())]
        rows.append(in_t3.groupby("ID_Equipe", as_index=False)["vote_at"].max().assign(Semestre=semester))
    return pd.concat(rows, ignore_index=True)


def clean_change_events(anchors: pd.DataFrame) -> pd.DataFrame:
    repositories = pd.read_parquet(COMMITS_PATH)[["Semestre", "ID_Equipe", "repository"]].drop_duplicates()
    anchored_repositories = anchors.merge(repositories, on=["Semestre", "ID_Equipe"], validate="one_to_one")
    minimum_offset, maximum_offset = CONFIG["rolling_end_day_range_relative_to_t3"]
    events = []
    for anchor in anchored_repositories.to_dict("records"):
        git_dir = PARENT_CACHE_DIR / f"{anchor['repository']}.git"
        branch = subprocess.check_output(["git", f"--git-dir={git_dir}", "symbolic-ref", "--short", "HEAD"], text=True).strip()
        raw_log = subprocess.check_output(["git", f"--git-dir={git_dir}", "log", branch, "--format=%H%x09%cI"], text=True)
        earliest = anchor["vote_at"].tz_convert("UTC") + pd.Timedelta(days=minimum_offset - CONFIG["rolling_window_days"])
        latest = anchor["vote_at"].tz_convert("UTC") + pd.Timedelta(days=maximum_offset)
        for line in raw_log.splitlines():
            commit_hash, committed_at_raw = line.split("\t", 1)
            committed_at = pd.Timestamp(committed_at_raw)
            if not earliest <= committed_at < latest:
                continue
            clean_paths = set()
            clean_churn = 0
            for change in _git_diff_paths(git_dir, commit_hash):
                path = change["file_path"]
                category = classify_file_category(path)["file_category"]
                if is_m4_measurement_path(path, category):
                    clean_paths.add(path)
                    clean_churn += (change["lines_added"] or 0) + (change["lines_deleted"] or 0)
            events.append({
                "Semestre": anchor["Semestre"],
                "ID_Equipe": anchor["ID_Equipe"],
                "committed_at": committed_at,
                "clean_churn": clean_churn,
                "clean_paths": clean_paths,
            })
    return pd.DataFrame(events)


t3_anchors = last_t3_evaluator_vote_anchors(PROJECT_ROOT)
assert len(t3_anchors) == CONFIG["expected_team_semesters"]
assert not t3_anchors.duplicated(CONFIG["team_key"]).any()
clean_events = clean_change_events(t3_anchors)
rolling_m4_rows = []
for anchor in t3_anchors.to_dict("records"):
    team_events = clean_events.loc[
        clean_events["Semestre"].eq(anchor["Semestre"])
        & clean_events["ID_Equipe"].eq(anchor["ID_Equipe"])
    ]
    t3_anchor = anchor["vote_at"].tz_convert("UTC")
    for day_offset in range(
        CONFIG["rolling_end_day_range_relative_to_t3"][0],
        CONFIG["rolling_end_day_range_relative_to_t3"][1] + 1,
        CONFIG["rolling_step_days"],
    ):
        window_end = t3_anchor + pd.Timedelta(days=day_offset)
        window_start = window_end - pd.Timedelta(days=CONFIG["rolling_window_days"])
        current = team_events.loc[team_events["committed_at"].ge(window_start) & team_events["committed_at"].lt(window_end)]
        changed_paths = set().union(*current["clean_paths"].tolist()) if not current.empty else set()
        rolling_m4_rows.append({
            "Semestre": anchor["Semestre"],
            "ID_Equipe": anchor["ID_Equipe"],
            "window_end_day_relative_to_t3": day_offset,
            "clean_churn_7d": int(current["clean_churn"].sum()),
            "unique_clean_path_n_7d": len(changed_paths),
        })
rolling_m4 = pd.DataFrame(rolling_m4_rows)
rolling_m4_cohort = (
    rolling_m4.groupby(["Semestre", "window_end_day_relative_to_t3"], as_index=False)
    .agg(
        team_n=("ID_Equipe", "size"),
        total_clean_churn_7d=("clean_churn_7d", "sum"),
        median_clean_churn_7d=("clean_churn_7d", "median"),
        median_unique_clean_path_n_7d=("unique_clean_path_n_7d", "median"),
        teams_with_clean_change_n=("clean_churn_7d", lambda values: int(values.gt(0).sum())),
    )
)
rolling_m4_cohort["teams_with_clean_change_share"] = rolling_m4_cohort["teams_with_clean_change_n"] / rolling_m4_cohort["team_n"]
assert len(rolling_m4) == CONFIG["expected_team_semesters"] * 71
assert rolling_m4.groupby(CONFIG["team_key"]).size().eq(71).all()
rolling_m4_cohort.loc[rolling_m4_cohort["window_end_day_relative_to_t3"].isin([-56, -49, -42, -21, -7, 0, 7])].sort_values(["Semestre", "window_end_day_relative_to_t3"])

### 4.1 Não redundância e decisão

M3 e M7 respondem “quando há atividade” e “como a autoria/atividade se distribui”; M4d responde “quanto código-fonte/teste muda” e “quantos caminhos limpos são atingidos”. `commit_n` não entra como saída de M4d: ele só permite localizar os diffs dentro da janela. A análise conjunta com M5 permanece um contraste temporal descritivo, sem correlação entre grãos incompatíveis.

In [ ]:
m4_temporal_decision = pd.DataFrame([
    ("M4d", "adopt", "7-day rolling clean churn and unique clean paths aligned to team T3 anchors"),
    ("M3", "retain separately", "author concentration and commit timing; no churn magnitude output"),
    ("M7", "retain separately", "binary repository inactivity; no clean churn or file-amplitude output"),
    ("M5", "contrast descriptively", "global qualitative corpus has incompatible analytical grain"),
    ("paper", "replace raw-churn trajectory", "report clean source/test churn and artifact policy version"),
], columns=["component", "decision", "reason"])
assert CONFIG["commit_count_is_primary"] is False
assert rolling_m4["clean_churn_7d"].ge(0).all()
assert rolling_m4["unique_clean_path_n_7d"].ge(0).all()
m4_temporal_decision

In [ ]:
m4_temporal_evidence_manifest = {
    "metric": "M4",
    "rq": detected_rq,
    "analysis_level": "team_semester_rolling_window_then_cohort",
    "git_source": CONFIG["git_source"],
    "timestamp_field": CONFIG["timestamp_field"],
    "anchor": CONFIG["primary_anchor"],
    "rolling_window_days": CONFIG["rolling_window_days"],
    "rolling_step_days": CONFIG["rolling_step_days"],
    "rolling_end_day_range_relative_to_t3": CONFIG["rolling_end_day_range_relative_to_t3"],
    "included_categories": sorted(INCLUDED_M4_CATEGORIES),
    "extension_allowlist_n": len(M4_EXTENSION_ALLOWLIST),
    "artifact_policy_version": ARTIFACT_POLICY_VERSION,
    "team_semester_n": int(t3_anchors.shape[0]),
    "daily_windows_n": int(rolling_m4.shape[0]),
    "inference": "descriptive_only",
}
assert m4_temporal_evidence_manifest["daily_windows_n"] == 14 * 71
m4_temporal_evidence_manifest

## 5. Relatório de cobertura da política de artefatos

Esta auditoria percorre todos os eventos de arquivo de todos os repositórios. Ela mostra o que entra e sai do M4 por categoria, extensão, repositório e prefixo de caminho. Duas filas são obrigatórias para revisão humana: caminhos `unknown` com alto churn e caminhos source/test cujo nome sugere cópia, backup, template ou arquivamento. A segunda fila não altera automaticamente a blacklist.